# Try settings on earlier dates

## Objective

Try model settings on earlier dates before using the final test.

Run notebooks 2 and 3 first. **Tuning** tries settings. **Cross-validation** tests each setting on earlier windows. **Refitting** trains again using the chosen settings and later history.

We use two settings and two windows: four tuning fits, one validation fit and one final refit, six fits altogether. MAE measures the average absolute error; smaller is better. The final test does not choose settings.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


## Step 2: Start the data engine

We will start a small local Spark session with two workers. Spark reads and checks the study data. UTC keeps date and time handling consistent.


In [ ]:
# Import the tools used in this step.
from pyspark.sql import SparkSession

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark may print startup messages. The cell should finish without an exception. No forecasts have been made yet.


## Step 3: Add two earlier test windows

We will update the study rules and enable a two-trial search. Each trial is one choice of model settings, tested on both earlier windows.


In [ ]:
# Import the tools used in this step.
import yaml

# Read the current study settings before updating them.
settings = yaml.safe_load((work / "study.yaml").read_text())
# Use earlier dates to choose settings, leaving the final test untouched.
settings["recipe"]["tuning_windows"] = [
    {"origin": "2024-01-14", "horizon": 7},
    {"origin": "2024-01-21", "horizon": 7},
]
# Save the updated settings for the registry to read.
_ = (work / "study.yaml").write_text(yaml.safe_dump(settings))
# Read the model settings before enabling the search.
profile = yaml.safe_load((work / "model.yaml").read_text())
# Try two settings with a fixed random seed.
profile["hpo"] = {"backend": "optuna", "max_trials": 2, "seed": 42}
profile["selection"] = {"stage": "tuning", "origins": "all", "metric": "mae"}
# Save the allowed values and the default constant prediction.
_ = (work / "model.yaml").write_text(yaml.safe_dump(profile))

### Understand the output

No output is expected. The study now has two tuning windows and the model profile asks for two trials. No search has run yet.


The study rules changed, so register again and use the new ID. Omit `fixed_parameters` to search.


## Step 4: Register the changed study and search

We will create the study ID for these new rules, then run tuning, validation and final testing. The saved path will point to this new run.


In [ ]:
# Import the tools used in this step.
import importlib.util

from autoforecast import register_study, run_study
from autoforecast.study_registration import StudyRegistry

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)


# Load the model from the file we wrote, not from hidden notebook state.
spec = importlib.util.spec_from_file_location("tutorial_constant", work / "models/constant.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
# Check the data and save an ID for this exact study.
study_id = register_study(work / "study.yaml", registry=registry)
_ = (work / "study-id.txt").write_text(study_id)
# Fit and evaluate the model, keeping all output files.
result = run_study(
    study_id,
    module.fit_predict,
    registry=registry,
    model_config=work / "model.yaml",
    tracking="none",
    publish=False,
)
# Remember this run so later notebooks can reopen it.
_ = (work / "latest-run.txt").write_text(str(result.path))
# Show the result folder and a few useful quantities, without internal details.
summary = result.summary()
print("Saved run:", result.path)
print(
    "Products: original =",
    summary["original_series"],
    "retained =",
    summary["retained_series"],
    "excluded =",
    summary["excluded_series"],
)
print("Validation MAE:", result.validation_metrics["mae"])
print("Final-test MAE:", result.testing_metrics["mae"])
print("Table publication:", summary["publication"]["status"])

### Understand the output

You should see a work estimate of six fits, the saved run folder, and product counts of 2 original, 2 retained and 0 excluded. Retained products are the ones included in the experiment. For this unchanged example, validation and final-test MAE are about 2.83 units per product/day. Smaller MAE means smaller average error. `not_requested` means we chose file output with `publish=False`; it is not a training failure. The path after `Saved run:` is the workflow folder to keep.


Keep the study, tuning and workflow folders together. The workflow refers to retained tuning evidence. Notebook 5 opens it without fitting again.


## Step 5: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark is now closed. Start a fresh kernel when the next notebook asks you to.
